# 🔥 Forest Fires — Regressão Linear e Ridge

**Dataset:** [Forest Fires (UCI Machine Learning Repository)](https://archive.ics.uci.edu/ml/datasets/forest+fires)  
**Objetivo:** Prever a **área queimada** (`area`) em incêndios florestais no Parque Natural de Montesinho, Portugal, com base em variáveis meteorológicas e índices FWI (Fire Weather Index).

---

## Variáveis do Dataset

| Coluna | Tipo | Descrição |
|--------|------|----------|
| `X` | int | Coordenada espacial X (1–9) |
| `Y` | int | Coordenada espacial Y (2–9) |
| `month` | str | Mês do ano (jan–dec) |
| `day` | str | Dia da semana (mon–sun) |
| `FFMC` | float | Fine Fuel Moisture Code (umidade de combustíveis finos) |
| `DMC` | float | Duff Moisture Code (umidade da camada orgânica) |
| `DC` | float | Drought Code (seca do solo profundo) |
| `ISI` | float | Initial Spread Index (velocidade inicial de propagação) |
| `temp` | float | Temperatura (°C) |
| `RH` | int | Umidade relativa do ar (%) |
| `wind` | float | Velocidade do vento (km/h) |
| `rain` | float | Chuva (mm/m²) |
| `area` | float | **Alvo** — Área queimada (ha) |

---

## 📦 1. Importações

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import scipy.stats as stats

# Estilo visual
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

print('✅ Bibliotecas carregadas com sucesso!')

---

## 📂 2. Carregamento dos Dados

In [ ]:
# ----- ajuste o caminho se necessário -----
CSV_PATH = 'forestfires.csv'

df = pd.read_csv(CSV_PATH)

print(f'Linhas: {df.shape[0]}  |  Colunas: {df.shape[1]}')
df.head()

---

## 🔍 3. Investigação dos Dados (EDA)

In [ ]:
# 3.1 Informações gerais
df.info()

In [ ]:
# 3.2 Estatísticas descritivas
df.describe().round(2)

In [ ]:
# 3.3 Valores ausentes
missing = df.isnull().sum()
print('Valores ausentes por coluna:')
print(missing[missing > 0] if missing.any() else '✅ Nenhum valor ausente encontrado.')

In [ ]:
# 3.4 Distribuição da variável alvo (area)
n_zero = int((df['area'] == 0).sum())
area_pos = df.loc[df['area'] > 0, 'area']
log_pos  = np.log(area_pos)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))

axes[0].hist(df['area'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('area (original)')
axes[0].set_xlabel('Área queimada (ha)'); axes[0].set_ylabel('Frequência')

axes[1].hist(np.log1p(df['area']), bins=40, color='tomato', edgecolor='white')
axes[1].set_title('log1p(area) — todos os registros')
axes[1].set_xlabel('log(1 + área)'); axes[1].set_ylabel('Frequência')
axes[1].annotate(f'{n_zero} registros\ncom area = 0', xy=(0.05, 230), xytext=(1.6, 190),
                 arrowprops=dict(arrowstyle='->', color='black'), fontsize=10)

axes[2].hist(log_pos, bins=25, color='mediumseagreen', edgecolor='white', density=True, label='area > 0')
xs = np.linspace(log_pos.min(), log_pos.max(), 200)
axes[2].plot(xs, stats.norm.pdf(xs, log_pos.mean(), log_pos.std()), color='black', linewidth=1.5, label='Normal ajustada')
axes[2].set_title(f'log(area) — só os {len(area_pos)} registros com area > 0')
axes[2].set_xlabel('log(área)'); axes[2].set_ylabel('Densidade'); axes[2].legend()

plt.suptitle('Variável alvo: a assimetria se resolve com o log; a massa em zero, não', y=1.03, fontweight='bold')
plt.tight_layout()
plt.show()

bc, lam = stats.boxcox(area_pos)
print(f'Registros com area = 0            : {n_zero} de {len(df)} ({n_zero/len(df)*100:.1f}%)')
print(f'Skewness de area (original)       : {df["area"].skew():.2f}')
print(f'Skewness de log1p(area), todos    : {np.log1p(df["area"]).skew():.2f}')
print(f'Skewness de log(area), area > 0   : {log_pos.skew():.2f}')
print(f'Shapiro-Wilk em log(area), area>0 : p = {stats.shapiro(log_pos).pvalue:.3f}')
print(f'Box-Cox em area > 0: lambda = {lam:.3f}  (lambda ≈ 0 equivale ao log)')

> **O que o gráfico mostra.** O painel do meio é o que normalmente se apresenta como "alvo transformado", e ele não parece uma normal porque não pode parecer: 247 dos 517 incêndios (47,8%) têm `area` exatamente igual a zero, e uma transformação monotônica leva 247 valores iguais em 247 valores iguais. Nenhum log, raiz ou Box-Cox move esse pico. O painel da direita isola os 270 registros com área positiva, e aí o `log(area)` é praticamente normal: skewness 0,30, Shapiro-Wilk sem rejeitar a normalidade (p ≈ 0,24) e Box-Cox escolhendo λ ≈ 0, que é o próprio log.
>
> Trata-se, portanto, de um alvo **semicontínuo com inflação de zeros**, e não de um alvo apenas assimétrico. No dataset original, `area = 0` significa área abaixo de 0,01 ha, ou seja, o incêndio aconteceu e foi registrado, mas ficou abaixo do limiar de medição. Isso sugere separar duas perguntas: *o incêndio passou do limiar?* (classificação) e, se passou, *quanto queimou?* (regressão). A seção 6.5 testa exatamente essa ideia.

In [ ]:
# 3.5 Distribuições das variáveis numéricas
num_cols = ['FFMC', 'DMC', 'DC', 'ISI', 'temp', 'RH', 'wind', 'rain']

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=30, color='slateblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel('Valor')
    axes[i].set_ylabel('Frequência')

plt.suptitle('Distribuições das Variáveis Preditoras Numéricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 3.6 Contagem das variáveis categóricas
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

month_order = ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']
day_order   = ['mon','tue','wed','thu','fri','sat','sun']

df['month'].value_counts().reindex(month_order).dropna().plot(kind='bar', ax=axes[0], color='mediumseagreen', edgecolor='white')
axes[0].set_title('Ocorrências por Mês', fontweight='bold')
axes[0].set_xlabel('Mês')
axes[0].set_ylabel('Contagem')
axes[0].tick_params(axis='x', rotation=0)

df['day'].value_counts().reindex(day_order).dropna().plot(kind='bar', ax=axes[1], color='darkorange', edgecolor='white')
axes[1].set_title('Ocorrências por Dia da Semana', fontweight='bold')
axes[1].set_xlabel('Dia')
axes[1].set_ylabel('Contagem')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# 3.7 Mapa de calor de correlações (variáveis numéricas)
df_log = df.copy()
df_log['log_area'] = np.log1p(df['area'])

corr_cols = num_cols + ['X', 'Y', 'log_area']
corr_matrix = df_log[corr_cols].corr()

plt.figure(figsize=(11, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, linewidths=0.5,
    cbar_kws={'label': 'Correlação de Pearson'}
)
plt.title('Matriz de Correlação (incluindo log_area)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

> **Leitura do mapa de calor: duas histórias diferentes.** A última linha, a de `log_area`, é a que interessa para a previsão, e ela é quase toda cinza: nenhuma variável passa de 0,07 em módulo. O restante do mapa conta outra história, a das relações entre os próprios preditores, e aí as cores aparecem. `DMC` e `DC` têm correlação de 0,68, o que faz sentido porque os dois medem umidade acumulada no solo, apenas em camadas e janelas de tempo diferentes. `temp` se correlaciona com `DC` (0,50), com `DMC` (0,47) e com `FFMC` (0,43), e se opõe a `RH` (−0,53), já que dias quentes tendem a ser secos. `FFMC` e `ISI` ficam em 0,53, porque o ISI é calculado a partir do FFMC e do vento. Por fim, `X` e `Y` se correlacionam em 0,54, sinal de que os incêndios se concentram numa diagonal da grade do parque.
>
> Essa estrutura é a **multicolinearidade** que o Ridge foi desenhado para tratar. Quando dois preditores carregam quase a mesma informação, o OLS tem liberdade para dividir o peso entre eles de forma arbitrária, inclusive dando sinais opostos a cada um; a penalização L2 limita essa liberdade e distribui o peso de maneira mais estável. É importante, porém, não confundir as duas histórias: a colinearidade explica por que os coeficientes do OLS serão instáveis, mas não cria sinal onde ele não existe. A linha de `log_area` continua cinza com ou sem regularização.

In [ ]:
# 3.8 Scatter plots: preditores vs log_area
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].scatter(df_log[col], df_log['log_area'], alpha=0.35, color='royalblue', edgecolors='none', s=25)
    m, b = np.polyfit(df_log[col], df_log['log_area'], 1)
    x_line = np.linspace(df_log[col].min(), df_log[col].max(), 100)
    axes[i].plot(x_line, m*x_line + b, color='tomato', linewidth=1.5)
    axes[i].set_title(f'{col} vs log_area', fontweight='bold')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('log(1+area)')

plt.suptitle('Relação entre Preditores e log(1+área)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 3.9 Boxplots por mês e dia
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

df_log['month_cat'] = pd.Categorical(df_log['month'], categories=month_order, ordered=True)
df_log['day_cat']   = pd.Categorical(df_log['day'],   categories=day_order,   ordered=True)

sns.boxplot(data=df_log.sort_values('month_cat'), x='month_cat', y='log_area',
            palette='tab10', ax=axes[0])
axes[0].set_title('log(1+area) por Mês', fontweight='bold')
axes[0].set_xlabel('Mês')
axes[0].set_ylabel('log(1+area)')

sns.boxplot(data=df_log.sort_values('day_cat'), x='day_cat', y='log_area',
            palette='Set2', ax=axes[1])
axes[1].set_title('log(1+area) por Dia da Semana', fontweight='bold')
axes[1].set_xlabel('Dia')
axes[1].set_ylabel('log(1+area)')

plt.tight_layout()
plt.show()

> **Leitura dos boxplots.** Em quase todos os meses, a caixa começa em zero, porque o primeiro quartil é zero: pelo menos um quarto dos incêndios de cada mês não atingiu 0,01 ha. Em março, junho e outubro, a própria mediana é zero, ou seja, mais da metade dos incêndios desses meses ficou abaixo do limiar. Agosto e setembro, que concentram dois terços dos registros, têm medianas de 0,44 e 0,86 em `log1p(area)`, pouco acima da mediana geral, e são também os meses com as caudas mais longas, pois os incêndios realmente grandes, incluindo o de 1 090 ha, acontecem no fim do verão.
>
> Dezembro destoa: a caixa inteira fica acima de 2,3, sem nenhum zero. Antes de concluir que dezembro é um mês de incêndios grandes, convém olhar quantos registros há ali: são nove, e quatro deles compartilham o mesmo dia da semana, a mesma temperatura, a mesma umidade e o mesmo vento, o que indica um único evento registrado em quatro células da grade. Janeiro, maio e novembro têm um ou dois registros cada, de modo que suas "caixas" são, na prática, um ou dois pontos. Os boxplots, portanto, ensinam duas coisas ao mesmo tempo: que a variação entre os meses com amostra suficiente é pequena, e que os meses com amostra insuficiente produzem figuras enganosas. É por esse motivo que, na preparação dos dados, esses meses serão agrupados numa única categoria.
>
> Os dias da semana são mais fáceis de ler: as medianas ficam entre 0,26 e 0,64, os terceiros quartis entre 1,8 e 2,3, e domingo é o único dia com mediana zero. Nada nesse painel sugere que o dia da semana ajude a prever a área queimada.

---

## 🛠️ 4. Preparação dos Dados

Com base na EDA, adotamos as seguintes decisões:

| Decisão | Justificativa |
|---------|---------------|
| Transformar `area` → `log1p(area)` | Estabiliza a escala dos valores positivos (skew de 12,8 cai para 1,2) e deixa o erro em escala multiplicativa, mais adequada a áreas que variam de 0,1 a 1 000 ha. A massa em zero continua lá e é tratada à parte na seção 6.5 |
| Agrupar os meses raros em `outros` | jan, may e nov têm 1 ou 2 registros; apr e dec, 9. Uma dummy ajustada em dois pontos vira ruído puro no OLS |
| Codificar `month` e `day` com **One-Hot Encoding** | São variáveis categóricas nominais sem ordem intrínseca |
| Manter `X` e `Y` como coordenadas numéricas | Representam localização espacial |
| Remover `rain` após análise | ~98% dos valores = 0; variância quase nula |
| Padronizar preditores numéricos (`StandardScaler`) | Requisito para Ridge (penalização coerente em todas as escalas) |

In [ ]:
# 4.1 Transformação logarítmica da variável alvo
df_proc = df.copy()
df_proc['log_area'] = np.log1p(df_proc['area'])

# 4.2 Remover 'rain' (variância quase nula) e 'area' (substituída por log_area)
print(f'Variância de rain: {df_proc["rain"].var():.4f}')
print(f'rain == 0: {(df_proc["rain"]==0).mean()*100:.1f}% dos registros')
df_proc = df_proc.drop(columns=['rain', 'area'])

# 4.3 Agrupar meses raros (menos de 10 registros) e aplicar One-Hot Encoding
contagem_mes = df_proc['month'].value_counts()
meses_raros  = contagem_mes[contagem_mes < 10].index.tolist()
print(f'\nMeses agrupados em "outros": {meses_raros}')
df_proc['month'] = df_proc['month'].replace(meses_raros, 'outros')

df_proc = pd.get_dummies(df_proc, columns=['month', 'day'], drop_first=True)

print(f'\nShape após pré-processamento: {df_proc.shape}')
df_proc.head(3)

In [ ]:
# 4.4 Separação entre features (X) e alvo (y)
TARGET = 'log_area'
X = df_proc.drop(columns=[TARGET])
y = df_proc[TARGET]

print('Features:', X.shape)
print('Alvo    :', y.shape)
print('\nColunas de features:')
print(X.columns.tolist())

In [ ]:
# 4.5 Divisão treino / teste (80/20, seed fixo para reprodutibilidade)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f'Treino : {X_train.shape[0]} amostras')
print(f'Teste  : {X_test.shape[0]} amostras')

In [ ]:
# 4.6 Padronização (fit apenas no treino para evitar data leakage)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('Padronização aplicada — média ≈ 0, desvio ≈ 1 no conjunto de treino.')

---

## 🤖 5. Modelos de Regressão

### 5.1 Regressão Linear (OLS)

In [ ]:
# Função auxiliar de avaliação
def avaliar(y_true, y_pred, label=''):
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f'  [{label}]  RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}')
    return {'RMSE': rmse, 'MAE': mae, 'R²': r2}

# Baseline: prever sempre a média do treino. Todo modelo precisa vencer isso.
y_pred_base_train = np.full(len(y_train), y_train.mean())
y_pred_base_test  = np.full(len(y_test),  y_train.mean())
print('=== Baseline (média do treino) ===')
m_base_train = avaliar(y_train, y_pred_base_train, 'Treino')
m_base_test  = avaliar(y_test,  y_pred_base_test,  'Teste ')

# Treino
lr = LinearRegression()
lr.fit(X_train_sc, y_train)

y_pred_lr_train = lr.predict(X_train_sc)
y_pred_lr_test  = lr.predict(X_test_sc)

print('\n=== Regressão Linear (OLS) ===')
m_lr_train = avaliar(y_train, y_pred_lr_train, 'Treino')
m_lr_test  = avaliar(y_test,  y_pred_lr_test,  'Teste ')

### 5.2 Ridge — Seleção automática de α via RidgeCV

In [ ]:
# Grade de alphas para busca
alphas = np.logspace(-3, 4, 200)   # 0.001 a 10000

ridge_cv = RidgeCV(alphas=alphas, cv=10, scoring='neg_root_mean_squared_error')
ridge_cv.fit(X_train_sc, y_train)

best_alpha = ridge_cv.alpha_
print(f'Melhor α encontrado pelo RidgeCV (10-fold): {best_alpha:.4f}')

In [ ]:
# Treino com o melhor alpha
ridge = Ridge(alpha=best_alpha)
ridge.fit(X_train_sc, y_train)

y_pred_ridge_train = ridge.predict(X_train_sc)
y_pred_ridge_test  = ridge.predict(X_test_sc)

print('=== Ridge Regression (α =', round(best_alpha, 4), ') ===')
m_ridge_train = avaliar(y_train, y_pred_ridge_train, 'Treino')
m_ridge_test  = avaliar(y_test,  y_pred_ridge_test,  'Teste ')

print(f'\nDesvio-padrão das previsões Ridge no teste : {y_pred_ridge_test.std():.3f}')
print(f'Desvio-padrão do alvo no teste             : {y_test.std():.3f}')
print('→ Um α dessa ordem encolhe quase todos os coeficientes para perto de zero: o modelo prevê quase sempre a média.')

### 5.3 Curva de regularização — impacto do α nos coeficientes

In [ ]:
alphas_plot = np.logspace(-3, 4, 100)
coefs_ridge = []

for a in alphas_plot:
    r = Ridge(alpha=a)
    r.fit(X_train_sc, y_train)
    coefs_ridge.append(r.coef_)

coefs_ridge = np.array(coefs_ridge)

plt.figure(figsize=(12, 5))
for i in range(coefs_ridge.shape[1]):
    plt.plot(alphas_plot, coefs_ridge[:, i], alpha=0.6, linewidth=1)
plt.axvline(best_alpha, color='black', linestyle='--', linewidth=1.5, label=f'α ótimo = {best_alpha:.3f}')
plt.xscale('log')
plt.xlabel('α (escala log)')
plt.ylabel('Valor do coeficiente')
plt.title('Trajetória dos Coeficientes Ridge em função de α', fontweight='bold')
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

---

## 📊 6. Avaliação e Comparação dos Modelos

In [ ]:
# 6.1 Tabela comparativa
resultados = pd.DataFrame({
    'Modelo'       : ['Baseline (média)', 'Linear (OLS)', 'Ridge'],
    'RMSE Treino'  : [m_base_train['RMSE'], m_lr_train['RMSE'], m_ridge_train['RMSE']],
    'RMSE Teste'   : [m_base_test['RMSE'],  m_lr_test['RMSE'],  m_ridge_test['RMSE']],
    'MAE Treino'   : [m_base_train['MAE'],  m_lr_train['MAE'],  m_ridge_train['MAE']],
    'MAE Teste'    : [m_base_test['MAE'],   m_lr_test['MAE'],   m_ridge_test['MAE']],
    'R² Treino'    : [m_base_train['R²'],   m_lr_train['R²'],   m_ridge_train['R²']],
    'R² Teste'     : [m_base_test['R²'],    m_lr_test['R²'],    m_ridge_test['R²']],
}).set_index('Modelo').round(4)

resultados

In [ ]:
# 6.2 Validação cruzada (10-fold) no conjunto completo
# O scaler entra num Pipeline para ser ajustado só na parte de treino de cada fold (sem vazamento).
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyRegressor

kf = KFold(n_splits=10, shuffle=True, random_state=42)

modelos_cv = {
    'Baseline (média)': DummyRegressor(strategy='mean'),
    'Linear (OLS)'    : make_pipeline(StandardScaler(), LinearRegression()),
    'Ridge'           : make_pipeline(StandardScaler(), Ridge(alpha=best_alpha)),
}
linhas = []
for nome, modelo in modelos_cv.items():
    sc = cross_val_score(modelo, X, y, cv=kf, scoring='neg_root_mean_squared_error')
    linhas.append({'Modelo': nome, 'CV RMSE médio': -sc.mean(), 'CV RMSE std': sc.std()})

cv_results = pd.DataFrame(linhas).set_index('Modelo').round(4)
print('=== Validação Cruzada 10-fold ===')
cv_results

In [ ]:
# 6.3 Gráfico: Predito vs Real (escala log)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, y_pred, title in zip(
    axes,
    [y_pred_lr_test, y_pred_ridge_test],
    ['Regressão Linear (OLS)', f'Ridge (α={best_alpha:.3f})']
):
    ax.scatter(y_test, y_pred, alpha=0.45, s=30, color='steelblue', edgecolors='none')
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfeito')
    ax.set_xlabel('Real (log_area)')
    ax.set_ylabel('Predito (log_area)')
    ax.set_title(title, fontweight='bold')
    r2 = r2_score(y_test, y_pred)
    ax.annotate(f'R² = {r2:.4f}', xy=(0.05, 0.92), xycoords='axes fraction', fontsize=11,
                bbox=dict(boxstyle='round,pad=0.3', fc='lightyellow', ec='gray'))
    ax.legend()

plt.suptitle('Valores Reais vs Preditos — Conjunto de Teste', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 6.4 Análise de resíduos
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

residuos_lr    = y_test - y_pred_lr_test
residuos_ridge = y_test - y_pred_ridge_test

# Resíduos vs Predito — Linear
axes[0, 0].scatter(y_pred_lr_test, residuos_lr, alpha=0.4, s=25, color='steelblue')
axes[0, 0].axhline(0, color='red', linestyle='--')
axes[0, 0].set_title('OLS — Resíduos vs Predito', fontweight='bold')
axes[0, 0].set_xlabel('Predito'); axes[0, 0].set_ylabel('Resíduo')

# Resíduos vs Predito — Ridge
axes[0, 1].scatter(y_pred_ridge_test, residuos_ridge, alpha=0.4, s=25, color='tomato')
axes[0, 1].axhline(0, color='red', linestyle='--')
axes[0, 1].set_title('Ridge — Resíduos vs Predito', fontweight='bold')
axes[0, 1].set_xlabel('Predito'); axes[0, 1].set_ylabel('Resíduo')

# Q-Q plot — Linear
stats.probplot(residuos_lr, dist='norm', plot=axes[1, 0])
axes[1, 0].set_title('OLS — Q-Q Plot dos Resíduos', fontweight='bold')

# Q-Q plot — Ridge
stats.probplot(residuos_ridge, dist='norm', plot=axes[1, 1])
axes[1, 1].set_title('Ridge — Q-Q Plot dos Resíduos', fontweight='bold')

plt.suptitle('Diagnóstico dos Resíduos — Conjunto de Teste', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

> **Leitura do diagnóstico de resíduos.** Os dois gráficos superiores mostram os resíduos contra os valores preditos, e a primeira coisa a notar é a escala horizontal: as previsões do OLS variam entre aproximadamente 0,1 e 1,9, e as do Ridge, entre 1,06 e 1,13. Como o Ridge prevê um valor quase constante, o gráfico dele reproduz a distribuição do próprio alvo deslocada para baixo: uma faixa horizontal de pontos em torno de −1,1, formada pelos incêndios com `area = 0`, cujo `log1p` vale zero, e uma nuvem acima dela, formada pelos positivos. No OLS a faixa inclina-se, porque o resíduo dos zeros é exatamente o negativo da previsão, mas a estrutura é a mesma.
>
> Num modelo bem especificado, os resíduos formariam uma nuvem sem padrão em torno de zero. Aqui o padrão é visível e tem uma causa conhecida: a faixa inferior é a massa de zeros da seção 3.4 reaparecendo. O que à primeira vista parece heterocedasticidade, isto é, variância do erro crescendo com o valor predito, é na verdade a mistura de duas populações que o modelo trata como uma só.
>
> Os gráficos Q-Q, na parte inferior, confirmam o diagnóstico. Se os resíduos fossem normais, os pontos acompanhariam a reta. Em vez disso, a cauda esquerda fica achatada, porque dezenas de resíduos têm praticamente o mesmo valor negativo (os zeros), e a cauda direita sobe acima da reta, porque os poucos incêndios muito grandes geram resíduos positivos que nenhuma previsão próxima da média consegue alcançar. Esse formato em "S" não é uma violação leve do pressuposto de normalidade que se corrige com uma transformação: é a assinatura de um alvo semicontínuo, e a resposta adequada é tratar as duas populações separadamente, como a seção 6.5 faz.

### 6.5 Modelo em duas partes (hurdle): o alvo com inflação de zeros tratado como ele é

A seção 3.4 mostrou que a dificuldade não é a assimetria, e sim a massa de 47,8% de zeros. A forma clássica de modelar um alvo assim é dividir o problema em duas etapas: uma **classificação** (o incêndio passou de 0,01 ha?) e, só para os que passaram, uma **regressão** em `log(area)`, que ali é quase normal. Se as variáveis meteorológicas tiverem algum sinal, ele deve aparecer em pelo menos uma das duas etapas.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score, accuracy_score

# Etapa 1 — classificação: area > 0 ?
y_bin = (df['area'] > 0).astype(int)
clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=3000))
p_cv = cross_val_predict(clf, X, y_bin, cv=kf, method='predict_proba')[:, 1]

print('=== Etapa 1: o incêndio passou de 0,01 ha? (regressão logística, 10-fold) ===')
print(f'  AUC                        : {roc_auc_score(y_bin, p_cv):.3f}   (0,5 = sorteio)')
print(f'  Acurácia (corte 0,5)       : {accuracy_score(y_bin, p_cv > 0.5):.3f}')
print(f'  Acurácia de chutar a classe majoritária: {max(y_bin.mean(), 1-y_bin.mean()):.3f}')

# Etapa 2 — regressão só nos positivos, alvo log(area)
mask_pos = (df['area'] > 0).values
X_pos    = X[mask_pos]
y_pos    = np.log(df.loc[mask_pos, 'area'])

print('\n=== Etapa 2: dado que queimou, quanto? (Ridge em log(area), só area > 0, 10-fold) ===')
print(f'  Desvio-padrão de log(area) nos positivos: {y_pos.std():.3f}')
for a in [1, 10, 100]:
    reg = make_pipeline(StandardScaler(), Ridge(alpha=a))
    yp  = cross_val_predict(reg, X_pos, y_pos, cv=kf)
    print(f'  α={a:>3}  RMSE={np.sqrt(mean_squared_error(y_pos, yp)):.3f}  R²={r2_score(y_pos, yp):.3f}')

# Correlação de cada variável numérica com cada uma das duas perguntas
num_cols = ['FFMC', 'DMC', 'DC', 'ISI', 'temp', 'RH', 'wind']
corr_duas = pd.DataFrame({
    'corr com (area > 0)'        : df[num_cols].corrwith(y_bin.astype(float)),
    'corr com log(area) | area>0': df.loc[mask_pos, num_cols].corrwith(y_pos),
}).round(3)
print('\nCorrelação de Pearson das variáveis meteorológicas com cada etapa:')
corr_duas

In [ ]:
# Proporção de incêndios com area > 0 por mês e por dia da semana
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

meses_ok = df['month'].value_counts()
meses_ok = meses_ok[meses_ok >= 10].index
frac_mes = df[df['month'].isin(meses_ok)].groupby('month')['area'].apply(lambda s: (s > 0).mean()).reindex(month_order).dropna()
frac_dia = df.groupby('day')['area'].apply(lambda s: (s > 0).mean()).reindex(day_order)

for ax, serie, titulo, cor in [(axes[0], frac_mes, 'por mês (meses com ≥ 10 registros)', 'mediumseagreen'),
                               (axes[1], frac_dia, 'por dia da semana', 'darkorange')]:
    serie.plot(kind='bar', ax=ax, color=cor, edgecolor='white')
    ax.axhline(y_bin.mean(), color='black', linestyle='--', linewidth=1, label=f'média geral = {y_bin.mean():.2f}')
    ax.set_ylim(0, 1); ax.set_ylabel('Proporção com area > 0'); ax.set_xlabel('')
    ax.set_title(f'Proporção de incêndios com área > 0 {titulo}', fontweight='bold')
    ax.tick_params(axis='x', rotation=0); ax.legend(loc='upper right')

plt.tight_layout()
plt.show()

> **Leitura.** Nenhuma das duas etapas encontra sinal. A classificação fica com AUC em torno de 0,54, praticamente um sorteio, e a regressão nos positivos tem R² negativo, ou seja, perde para a média. As correlações ficam todas abaixo de 0,1 em módulo, e a proporção de incêndios com área mensurável gira em torno de 50% em todo mês e em todo dia da semana. O modelo em duas partes é a abordagem estatisticamente correta para este alvo, e justamente por isso o resultado dele é informativo: as variáveis disponíveis descrevem as condições no momento da ignição, mas a área final depende do que não está na tabela — tempo de resposta do combate, relevo, carga de combustível.

---

## 🧮 7. Importância dos Coeficientes

In [ ]:
feature_names = X.columns.tolist()

coef_df = pd.DataFrame({
    'Feature'   : feature_names,
    'Coef OLS'  : lr.coef_,
    'Coef Ridge': ridge.coef_
}).set_index('Feature').sort_values('Coef Ridge', key=abs, ascending=False)

coef_df.head(15)

In [ ]:
# Gráfico — Top 15 features por magnitude (Ridge)
top15 = coef_df.head(15)

fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=True)

colors_ols   = ['steelblue' if v >= 0 else 'salmon'       for v in top15['Coef OLS']]
colors_ridge = ['mediumseagreen' if v >= 0 else 'tomato'  for v in top15['Coef Ridge']]

axes[0].barh(top15.index, top15['Coef OLS'], color=colors_ols, edgecolor='white')
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title('Coeficientes — OLS', fontweight='bold')
axes[0].set_xlabel('Valor do coeficiente')
axes[0].invert_yaxis()

axes[1].barh(top15.index, top15['Coef Ridge'], color=colors_ridge, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title(f'Coeficientes — Ridge (α={best_alpha:.3f})', fontweight='bold')
axes[1].set_xlabel('Valor do coeficiente')

plt.suptitle('Top 15 Features — Comparação OLS vs Ridge', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

> Repare na escala dos dois painéis: os coeficientes Ridge são de 10 a 20 vezes menores que os do OLS, e `DC` troca de sinal entre um modelo e outro. Quando a ordem e o sinal dos coeficientes mudam tanto com a regularização, eles não estão medindo efeito nenhum; estão medindo ruído de amostragem.

---

## 📝 8. Interpretação dos Resultados

### 8.1 Qualidade do ajuste

Os três modelos da tabela 6.1 ficam colados no baseline que prevê sempre a média. Na validação cruzada, o Ridge empata com esse baseline (diferença na terceira casa decimal) e o OLS fica atrás dele. No holdout de 20% a ordem se inverte, com o OLS ligeiramente à frente, e essa discordância entre as duas avaliações é, por si só, um sinal de que a diferença entre os modelos está dentro do ruído. O R² de teste fica em torno de zero em todos os casos. Isso não é um erro de implementação, e sim uma propriedade do dataset: a literatura que usou o Forest Fires desde o artigo original (Cortez e Morais, 2007) reporta a mesma dificuldade, inclusive com modelos não lineares.

### 8.2 OLS vs Ridge

| Aspecto | OLS | Ridge |
|---------|-----|-------|
| Regularização | Nenhuma | L2 (penaliza coeficientes grandes) |
| Sensível a multicolinearidade | Sim | Reduzida |
| Risco de overfitting | Maior | Menor |
| Coeficientes | Podem ser grandes | Encolhidos em direção a 0 |

O α escolhido pelo `RidgeCV` bateu no teto da grade (10⁴), e isso merece ser dito com clareza: com uma penalização desse tamanho, as previsões do Ridge têm desvio-padrão de 0,01, contra 1,5 do alvo. A validação cruzada não encontrou um equilíbrio entre viés e variância; ela pediu o maior α disponível porque desligar as variáveis é o que menos piora. É exatamente o que a regularização deve fazer quando as features são ruído, e por isso o Ridge não perde para o baseline enquanto o OLS perde.

### 8.3 Coeficientes: por que não interpretar

A seção 7 mostra que a ordem dos coeficientes não é estável. No OLS, os maiores são `DMC`, `month_outros` e `month_sep`, e `DC` aparece com sinal negativo; no Ridge, todos os coeficientes ficam abaixo de 0,005 e `DC` passa a positivo. Antes do agrupamento dos meses raros, o topo do OLS era `month_dec`, uma dummy ajustada em nove registros, o que mostra como essa lista depende de detalhes do pré-processamento. Variáveis que a intuição colocaria à frente, como `temp`, `ISI` e `RH`, aparecem com correlação abaixo de 0,1 com o alvo e não se destacam em nenhum dos dois modelos. Nessas condições, atribuir um efeito a "dias mais quentes" ou "mais seca" seria ler o ruído como se fosse sinal.

### 8.4 Diagnóstico dos resíduos

Como o Ridge prevê um valor quase constante, o gráfico de resíduos contra predito reproduz a distribuição do próprio alvo: uma faixa horizontal de pontos em torno de −1 (os incêndios com `area = 0`, cujo `log1p` é zero) e uma nuvem acima dela (os positivos). A "heterocedasticidade" e a cauda do Q-Q plot são a inflação de zeros da seção 3.4 reaparecendo nos resíduos, não uma violação leve de pressuposto.

### 8.5 Conclusão

> 1. A transformação logarítmica está correta, mas resolve só metade do problema: ela normaliza os valores positivos e não tem como tocar na massa de 47,8% de zeros. O alvo é semicontínuo, e o tratamento adequado é o modelo em duas partes da seção 6.5.
> 2. Mesmo com esse tratamento, as variáveis meteorológicas e os índices FWI não explicam nem **se** o incêndio ultrapassa 0,01 ha nem **quanto** ele queima quando ultrapassa. O melhor modelo linear é, na prática, a média.
> 3. Entre OLS e Ridge, o Ridge é preferível por não piorar o baseline e por manter os coeficientes pequenos e estáveis, mas a diferença é marginal.
> 4. O ganho real viria de dados que não estão na tabela: tempo até o início do combate, tipo de vegetação, declividade, e uma variável alvo que separasse área medida de área abaixo do limiar.

In [ ]:
# 8.6 Resumo final compacto
print('=' * 64)
print('  RESUMO FINAL — Forest Fires Regression')
print('=' * 64)
print(f'  Dataset           : {df.shape[0]} amostras, {df.shape[1]} variáveis originais')
print(f'  Zeros no alvo     : {n_zero} ({n_zero/len(df)*100:.1f}%) — alvo semicontínuo')
print(f'  Alvo (após transf): log1p(area)')
print(f'  Features usadas   : {X.shape[1]}')
print(f'  Split             : 80% treino / 20% teste (random_state=42)')
print()
print('  Modelo                 RMSE Teste   MAE Teste   R² Teste')
print('  ' + '-' * 56)
print(f'  Baseline (média)       {m_base_test["RMSE"]:.4f}       {m_base_test["MAE"]:.4f}      {m_base_test["R²"]:.4f}')
print(f'  OLS                    {m_lr_test["RMSE"]:.4f}       {m_lr_test["MAE"]:.4f}      {m_lr_test["R²"]:.4f}')
print(f'  Ridge (α={best_alpha:.0f})         {m_ridge_test["RMSE"]:.4f}       {m_ridge_test["MAE"]:.4f}      {m_ridge_test["R²"]:.4f}')
print()
print(f'  Duas partes — AUC (area>0): {roc_auc_score(y_bin, p_cv):.3f}   R² (log area | >0): {r2_score(y_pos, yp):.3f}')
print('=' * 64)